In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

In [3]:
import numpy as np
import pylab as plt

# Training

In [4]:
from vimms.Common import POSITIVE
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.features import obs_to_dfs
from vimms_gym.evaluation import evaluate
from vimms_gym.policy import fullscan_policy, random_policy, topN_policy, best_ppo_policy

/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/statsmodels/compat/pandas.py:61: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import Int64Index as NumericIndex


In [5]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)

In [6]:
n_chemicals = (500, 2000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E5, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE
noise_density = 0.3
noise_max_val = 1e4

In [9]:
mzml_filename = 'Beer_multibeers_1_fullscan1.mzML'
mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                       min_log_intensity=min_log_intensity,
                                       max_log_intensity=max_log_intensity)

2022-03-15 09:29:43.297 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans
2022-03-15 09:29:45.217 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 1109 scans


In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
    },
    'noise': {
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'N': N,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
        'min_ms1_intensity': min_ms1_intensity
    }
}

# Controller test

Generate some chemical sets

In [11]:
num_episodes = 5

In [12]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(num_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0


2022-03-15 09:30:54.650 | DEBUG    | vimms.Chemicals:sample:468 - Sampled rt and intensity values and chromatograms


1


2022-03-15 09:30:55.597 | DEBUG    | vimms.Chemicals:sample:468 - Sampled rt and intensity values and chromatograms


2


2022-03-15 09:30:56.615 | DEBUG    | vimms.Chemicals:sample:468 - Sampled rt and intensity values and chromatograms


3


2022-03-15 09:30:57.487 | DEBUG    | vimms.Chemicals:sample:468 - Sampled rt and intensity values and chromatograms


4


2022-03-15 09:30:58.400 | DEBUG    | vimms.Chemicals:sample:468 - Sampled rt and intensity values and chromatograms


In [13]:
out_dir = 'evaluation_results'
max_peaks = 20

## Fullscan controller

In [14]:
env = DDAEnv(max_peaks, params)
for i in range(num_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action = fullscan_policy(observation)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = 'fullscan_%d.mzML' % i
            env.write_mzML(out_dir, out_file)     
            print(evaluate(env))
            break

Episode 0 finished after 2001 timesteps with reward 0
(0.0, 0.0)
Episode 1 finished after 2001 timesteps with reward 0
(0.0, 0.0)
Episode 2 finished after 2001 timesteps with reward 0
(0.0, 0.0)
Episode 3 finished after 2001 timesteps with reward 0
(0.0, 0.0)
Episode 4 finished after 2001 timesteps with reward 0
(0.0, 0.0)


## Random controller

In [15]:
env = DDAEnv(max_peaks, params)
for i in range(num_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action = random_policy(observation)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = 'random_%d.mzML' % i
            env.write_mzML(out_dir, out_file) 
            print(evaluate(env))
            break

Episode 0 finished after 3812 timesteps with reward -21729.586001340893
(0.6335944299390774, 0.571564317148165)
Episode 1 finished after 3789 timesteps with reward -23484.33203256017
(0.825136612021858, 0.7661929845131693)
Episode 2 finished after 3787 timesteps with reward -22452.83311795788
(0.7728, 0.7163327181538107)
Episode 3 finished after 3814 timesteps with reward -20293.03498355877
(0.5630580357142857, 0.4899089828506281)
Episode 4 finished after 3819 timesteps with reward -22633.89250186244
(0.6753812636165577, 0.6206137365889307)


## Top-N controller

In [21]:
N = 10
env = DDAEnv(max_peaks, params)
for i in range(num_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        action = topN_policy(observation, N)
        observation, reward, done, info = env.step(action)
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = 'topN_%d.mzML' % i
            env.write_mzML(out_dir, out_file)
            print(evaluate(env))
            break

Episode 0 finished after 2647 timesteps with reward 2551.8002652474643
(0.6579634464751958, 0.5643439904643799)
Episode 1 finished after 2644 timesteps with reward 407.93422650390403
(0.8542805100182149, 0.7339769198913634)
Episode 2 finished after 2640 timesteps with reward 1246.6387407745865
(0.792, 0.6862590375179966)
Episode 3 finished after 2660 timesteps with reward 4373.140418904084
(0.5904017857142857, 0.4913924215863884)
Episode 4 finished after 2643 timesteps with reward 2286.219493754959
(0.6993464052287581, 0.6048921430632402)


## Model-based Policy (PPO)

In [14]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

In [15]:
model_name = 'PPO'
fname = 'results/model_%s' % model_name
model = PPO.load(fname)

In [329]:
actions = []
rewards = []
env = DDAEnv(max_peaks, params)
for i in range(num_episodes):
    chems = chem_list[i]
    observation = env.reset(chems=chems)
    done = False
    num_steps = 0
    episode_reward = 0
    
    while not done:
        # action = best_ppo_policy(observation, model)
        action, _states = model.predict(observation, deterministic=True)
        observation, reward, done, info = env.step(action)
        actions.append(action)
        rewards.append(reward)
        
        num_steps += 1
        episode_reward += reward
        if done:
            print(f'Episode {i} finished after {num_steps} timesteps with reward {episode_reward}')
            out_file = '%s_%d.mzML' % (model_name, i)
            env.write_mzML(out_dir, out_file)
            print(evaluate(env))
            break
            
    break

Episode 0 finished after 2028 timesteps with reward -5500
(0.0, 0.0)


In [ ]:
plt.scatter(range(len(actions)), actions)
plt.scatter(range(len(rewards)), rewards)

### Stepping through

In [18]:
env = DDAEnv(max_peaks, params)
i = 0
chems = chem_list[i]
observation = env.reset(chems=chems)
scan_df, count_df = obs_to_dfs(observation)
scan_df, count_df

(    mzs  rts  intensities  fragmented  excluded  above_min_intensity
 0   0.0  0.0          0.0         0.0       0.0                  0.0
 1   0.0  0.0          0.0         0.0       0.0                  0.0
 2   0.0  0.0          0.0         0.0       0.0                  0.0
 3   0.0  0.0          0.0         0.0       0.0                  0.0
 4   0.0  0.0          0.0         0.0       0.0                  0.0
 5   0.0  0.0          0.0         0.0       0.0                  0.0
 6   0.0  0.0          0.0         0.0       0.0                  0.0
 7   0.0  0.0          0.0         0.0       0.0                  0.0
 8   0.0  0.0          0.0         0.0       0.0                  0.0
 9   0.0  0.0          0.0         0.0       0.0                  0.0
 10  0.0  0.0          0.0         0.0       0.0                  0.0
 11  0.0  0.0          0.0         0.0       0.0                  0.0
 12  0.0  0.0          0.0         0.0       0.0                  0.0
 13  0.0  0.0       

In [325]:
scan_df, count_df = obs_to_dfs(observation)
action = best_ppo_policy(observation, model)
observation, reward, done, info = env.step(action)
print(reward)
scan_df, count_df

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[[0.04508271 0.05007606 0.04368455 0.04231093 0.03930987 0.03945092
  0.0409115  0.03998115 0.03572691 0.05935638 0.04411092 0.03947304
  0.04251049 0.04856737 0.0457016  0.03001561 0.03950632 0.04177516
  0.05155218 0.04815152 0.13274488]]
[[0.04508271 0.05007606 0.04368455 0.04231093 0.03930987 0.03945092
  0.0409115  0.03998115 0.03572691 0.05935638 0.04411092 0.03947304
  0.04251049 0.04856737 0.0457016  0.03001561 0.03950632 0.04177516
  0.05155218 0.04815152 0.13274488]]
20
0


(           mzs    rts  intensities  fragmented  excluded  above_min_intensity
 0   276.698211  307.0    22.997404         0.0       0.0                  1.0
 1   104.668533  307.0    22.193166         0.0       0.0                  1.0
 2   118.123987  307.0    22.087348         0.0       0.0                  1.0
 3   149.245642  307.0    21.935598         0.0       0.0                  1.0
 4   258.517678  307.0    21.844502         0.0       0.0                  1.0
 5   258.021983  307.0    21.718603         0.0       0.0                  1.0
 6   252.503237  307.0    21.568935         0.0       0.0                  1.0
 7   189.849039  307.0    21.374249         0.0       0.0                  1.0
 8   227.300324  307.0    21.321385         0.0       0.0                  1.0
 9   118.219844  307.0    21.098338         0.0       0.0                  1.0
 10  144.255469  307.0    20.928970         0.0       0.0                  1.0
 11  118.697728  307.0    20.885306         0.0     